In [1]:
import json
import pandas as pd

## 1. Lecture et normalisation JSON

Chargement de `restaurants.json` et conversion en DataFrame avec `pd.json_normalize()`, en conservant les métadonnées `ville` et `pays`.

In [2]:
with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
df

,nom,adresse,type,notes,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie


## Challenge 1 — Normalisation d'un JSON imbriqué

Extraction et analyse des restaurants de plusieurs villes italiennes à partir du fichier `villes.json`.

### Objectifs
- Aplatir un JSON imbriqué avec `pd.json_normalize()`.
- Conserver les informations des villes avec `meta`.
- Calculer la colonne `note_moyenne` à partir de `notes`.
- Identifier la ville ayant la meilleure moyenne avec `villes.nom`.
- Compter les restaurants par catégorie avec `type`.

Étape 1 — explorer sans pandas

In [3]:
with open("../data/villes.json", encoding="utf-8") as f:
    data = json.load(f)

print("Nombre de villes :", len(data["villes"]))

print("La deuxième ville est :", data["villes"][1]["nom"])

nombre_restaurant_Syracuse = next(
    (len(x["restaurants"]) for x in data["villes"] if x["nom"] == "Syracuse"),
    None
)
print("Nombre de restaurant à Syracuse :", nombre_restaurant_Syracuse)

for villes in data["villes"]:
    if villes["nom"] == "Palerme":
        if villes["restaurants"]:
            print("Premier restaurant à Palerme :",
                villes["restaurants"][0]["nom"]
            )
        else:
            print("Aucun Restaurant à Palerme")
        break

Nombre de villes : 3
La deuxième ville est : Palerme
Nombre de restaurant à Syracuse : 2
Premier restaurant à Palerme : Antica Focacceria


Étape 2 — compter à la main

In [4]:
for villes in data["villes"]:
    print(villes["nom"], ":", len(villes.get("restaurants", [])))

Catane : 3
Palerme : 2
Syracuse : 2


Étape 3 — tout mettre à plat

In [5]:
df = pd.json_normalize(
    data,
    record_path=["villes", "restaurants"],
    meta=[
        ["villes", "nom"],
        ["villes", "region"],
        ["villes", "habitants"]
    ]
)

Étape 4 — exploiter

In [6]:
df["note_moyenne"] = df["notes"].apply(lambda x: (sum(x) / len(x)))

print("Ville ayant la meilleure moyenne des restaurants")
moyennes_par_ville = df.groupby("villes.nom")["note_moyenne"].mean().round(2)
meilleure_ville = moyennes_par_ville.nlargest(1)
nom_ville = meilleure_ville.index.item()
note = meilleure_ville.iloc[0]
print(nom_ville, ":", note)

print()
print("Ville ayant la meilleure moyenne des évaluations")
df_notes = df.explode("notes")
df_notes["notes"] = df_notes["notes"].astype("float")

moyennes_evaluations_par_ville = df_notes.groupby("villes.nom")["notes"].mean()
moyennes_evaluations_par_ville = moyennes_evaluations_par_ville.round(2)
meilleure_ville = moyennes_evaluations_par_ville.nlargest(1)
nom_ville = meilleure_ville.index.item()
note = meilleure_ville.iloc[0]
print(nom_ville, ":", note)

print()
print("Types de restaurant")
print(df.groupby("type").size())


Ville ayant la meilleure moyenne des restaurants
Syracuse : 4.47

Ville ayant la meilleure moyenne des évaluations
Syracuse : 4.64

Types de restaurant
type
glacier      2
poisson      2
trattoria    3
dtype: int64


## Challenge 2 — Interroger une API en série

Interrogation de l'API Adresse pour récupérer et analyser les résultats de géocodage de cinq adresses françaises.

### Objectifs
- Effectuer plusieurs requêtes HTTP avec `requests`.
- Gérer les erreurs réseau et les résultats manquants.
- Extraire les champs `label`, `score`, `postcode` et `city`.
- Construire un DataFrame avec `pd.DataFrame()`.
- Évaluer la pertinence des résultats à partir du score de correspondance.

In [7]:
import requests
import time

_HEADERS = {"User-Agent": "bootcamp-simplon-da/1.0 (antoine.preney@orange.fr)"}
_URL = "https://api-adresse.data.gouv.fr/search/"

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

Étape 1 — une seule, d'abord

In [8]:
adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

parametres = {}
parametres["q"] = adresses[0]
parametres["limit"] = 1

retour_serveur = interroger(_URL, parametres, _HEADERS)
print("Label :", retour_serveur["features"][0]["properties"]["label"])
print("Score :", retour_serveur["features"][0]["properties"]["score"])

Label : 20 Avenue de Ségur 75007 Paris
Score : 0.9716281818181818


Étape 2 — les cinq

In [ ]:
resultats_geocodage = []
for adresse in adresses:
    resultat_adresse = {}
    resultat_adresse["adresse"] = adresse

    parametres = {}
    parametres["q"] = adresse
    parametres["limit"] = 1
    retour_serveur = interroger(_URL, parametres, _HEADERS)

    if retour_serveur is not None:
        proprietes = (
            retour_serveur["features"][0].get("properties", {})
            if retour_serveur["features"]
            else {}
        )

        resultat_adresse["label"] = proprietes.get("label", None)
        resultat_adresse["score"] = proprietes.get("score", None)
        resultat_adresse["postcode"] = proprietes.get("postcode", None)
        resultat_adresse["city"] = proprietes.get("city", None)
    resultats_geocodage.append(resultat_adresse)
    print(adresse, ": traitée")
    time.sleep(1)

20 avenue de Ségur 75007 Paris : traitée
Grand Place, Lille : traitée
1 rue de la Paix, Paris : traitée
place Bellecour, Lyon : traitée
zzzzz qwerty 99999 : traitée


Étape 3 — en DataFrame

In [10]:
df_geocodage = pd.DataFrame(resultats_geocodage)
df_geocodage

,adresse,label,score,postcode,city
0,20 avenue de Ségur 75007 Paris,20 Avenue de Ségur 75007 Paris,0.971628,75007,Paris
1,"Grand Place, Lille",Place de Gand 59800 Lille,0.568244,59800,Lille
2,"1 rue de la Paix, Paris",1 Rue de la Paix 75002 Paris,0.964060,75002,Paris
3,"place Bellecour, Lyon",Place Bellecour 69002 Lyon,0.967367,69002,Lyon
4,zzzzz qwerty 99999,NaN,NaN,NaN,NaN


Étape 4 — lire le résultat

**1. Combien d'adresses ont été trouvées ?**

4 adresses sur 5.

**2. Laquelle a le score le plus bas parmi celles trouvées, et pourquoi à ton avis ?**

« Grand Place, Lille » avec un score de `0.568244`, car l'API a retourné « Place de Gand 59800 Lille », qui ne correspond pas à l'adresse demandée.

**3. Le label renvoyé correspond-il toujours à ce que tu avais demandé ?**

Non.

## Challenge 3 — Géocodage et cartographie des restaurants de Catane

Géocodage de restaurants italiens avec l'API Nominatim et visualisation de leurs positions sur une carte interactive avec Folium.

### Objectifs
- Charger et normaliser les données de `restaurants.json`.
- Calculer la colonne `note_moyenne` à partir de `notes`.
- Géocoder les adresses avec Nominatim en respectant ses limites d'utilisation.
- Ajouter les coordonnées `lat` et `lon` au DataFrame.
- Afficher les restaurants sur une carte interactive avec `folium`.
- Exporter les résultats dans `restaurants_geocodes.csv`.

Étape 1 — du JSON au DataFrame

In [11]:
with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

df_restaurants = pd.json_normalize(
    data,
    record_path="restaurants",
    meta=["ville", "pays"]
)

Étape 2 — la note moyenne

In [12]:
df_restaurants["note_moyenne"] = df_restaurants["notes"].apply(
    lambda x: sum(x) / len(x)
).round(2)
df_restaurants = df_restaurants.drop(["notes"], axis=1)

Étape 3 — géocoder

In [13]:
def geocoder_adresse(adresse):
    try:
        url = "https://nominatim.openstreetmap.org/search"
        parametres = {
            "q": adresse,
            "format": "json",
            "limit": 1
        }
        retour_serveur = interroger(url, parametres, _HEADERS)

        return float(retour_serveur[0]['lat']), float(retour_serveur[0]['lon'])

    except (IndexError, TypeError, KeyError):
        return None, None

    finally:
        time.sleep(1)

s_coordonnees = df_restaurants["adresse"].apply(geocoder_adresse)
df_coordonnees = pd.DataFrame(s_coordonnees.tolist(), columns=['lat', 'lon'])
df_restaurants[["lat", "lon"]] = df_coordonnees

Étape 4 — la carte

In [ ]:
import folium

carte = None

for restaurant in df_restaurants.itertuples():
    popup = (
        f"<b>{restaurant.nom}</b>"
        f"<hr>"
        f"Note moyenne : <b>{restaurant.note_moyenne:.2f} / 5</b>"
    )

    popup = folium.Popup(popup, max_width=250)

    point = [restaurant.lat, restaurant.lon]
    if carte is None:
        carte = folium.Map(location=point, zoom_start=14)

    folium.Marker(location=point, popup=popup).add_to(carte)

carte

Étape 5 — garder le résultat

In [15]:
# df_restaurants = df_restaurants.drop(["type", "ville", "pays"], axis=1)
df_restaurants.to_csv("../data/restaurants_geocodes.csv", index=False)